# Prediction-only — Frozen Hybrid HGB (52 features)

Put these files in the same folder as this notebook:

```text
project_folder/
├── Prediction_only_AnimalId.ipynb
├── Dataset_for_model_buliding_2.csv
├── PracticeTest.csv
└── xgboost_cache/
    └── hybrid_hgb_top160_final.joblib
```

`hybrid_hgb_top160_features.json` is **optional** because the feature names and order are normally retained in the frozen model. Run **Kernel → Restart & Run All**. The final cell produces `PracticeTest_AnimalId_predictions.csv`, with exactly `AnimalId,MilkingId` and original test-row order.

*Model details*: original Global Top-320 candidate generation, Top-160 ranking, and 52-feature frozen HGB. The PracticeTest file lacks lactation/DIM/status inputs, so auxiliary estimators fill these from the observed features. The final HGB itself is never retrained. Full-data inference may be computationally intensive; checkpoint parts allow resuming.

In [ ]:
# -*- coding: utf-8 -*-
"""AnimalId inference: frozen 52-feature Hybrid HGB + original feature engineering.

Folder layout (default):
  predict_animalid.py
  Dataset_for_model_buliding_2.csv
  PracticeTest.csv
  xgboost_cache/hybrid_hgb_top160_final.joblib

Optional: xgboost_cache/hybrid_hgb_top160_features.json
Run: python predict_animalid.py

Requires: pandas, numpy, scikit-learn, joblib
"""
from __future__ import annotations

import argparse
import csv
import gc
import hashlib
import io
import json
import os
from contextlib import redirect_stdout
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
MODEL_COLUMNS = [
    'AnimalId', 'LactationNumber', 'DaysInMilk', 'ReproductionStatus',
    'EventDate', 'Avgmilkflow', 'Flow30_60Session',
    'YieldFirst2Min_Session', 'YieldSession', 'DurationSession_sec',
]
DIM_BIN_WIDTH = 7
REFERENCE_ROWS_PER_GROUP = 250
GLOBAL_MAX_CANDIDATES = 320
MODEL_TOP_K = 160
PROJECTED_RANK_PENALTY_DAYS = 7
DIM_TOLERANCE_DAYS = 30
MILK_OBSERVED_COLS = [
    'Avgmilkflow', 'Flow30_60Session', 'YieldFirst2Min_Session',
    'YieldSession', 'DurationSession_sec',
]





In [ ]:

# ============================================================
# 3. Structural, distance, and reference helper functions
# ============================================================

BASE_NUMERIC_FEATURES = [
    "DaysInMilk",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
]


def event_day_from_origin(frame, origin_date):
    return (
        pd.to_datetime(frame["EventDate"], errors="coerce") - origin_date
    ).dt.days.astype("float32")


def lactation_start_day(frame, origin_date):
    event_date = pd.to_datetime(frame["EventDate"], errors="coerce")
    lactation_start = (
        event_date
        - pd.to_timedelta(frame["DaysInMilk"], unit="D")
    )
    return (lactation_start - origin_date).dt.days.astype("float32")


def build_structural_profiles(
    labelled_frame,
    class_to_number,
    origin_date,
    include_projection=True,
):
    temp = labelled_frame[
        ["AnimalId", "LactationNumber", "EventDate", "DaysInMilk"]
    ].copy()

    temp["EventDay"] = event_day_from_origin(temp, origin_date)
    temp["LactationStartDay"] = lactation_start_day(temp, origin_date)

    temp = temp.dropna(
        subset=[
            "AnimalId",
            "LactationNumber",
            "DaysInMilk",
            "EventDay",
            "LactationStartDay",
        ]
    )

    observed = (
        temp.groupby(["AnimalId", "LactationNumber"], observed=True)
        .agg(
            LactationStartDay=("LactationStartDay", "median"),
            LastEventDay=("EventDay", "max"),
            LastDaysInMilk=("DaysInMilk", "max"),
        )
        .reset_index()
    )

    observed["ClassId"] = (
        observed["AnimalId"].astype(object).map(class_to_number)
    )
    observed = observed.loc[observed["ClassId"].notna()].copy()

    observed["ClassId"] = observed["ClassId"].astype(np.int32)
    observed["LactationNumber"] = (
        observed["LactationNumber"].round().astype(np.int16)
    )

    for col in ["LactationStartDay", "LastEventDay", "LastDaysInMilk"]:
        observed[col] = observed[col].round().astype(np.int32)

    observed["IsProjected"] = False

    interval_rows = []
    for class_id, group in observed.groupby("ClassId", sort=False):
        group = group.sort_values("LactationNumber")
        lactations = group["LactationNumber"].to_numpy()
        starts = group["LactationStartDay"].to_numpy()

        for i in range(1, len(group)):
            if lactations[i] - lactations[i - 1] == 1:
                interval = starts[i] - starts[i - 1]
                if 250 <= interval <= 550:
                    interval_rows.append(
                        {
                            "ClassId": int(class_id),
                            "IntervalDays": int(interval),
                        }
                    )

    if interval_rows:
        interval_df = pd.DataFrame(interval_rows)
        global_interval = int(round(interval_df["IntervalDays"].median()))
        animal_interval = (
            interval_df.groupby("ClassId")["IntervalDays"].median().to_dict()
        )
    else:
        global_interval = 365
        animal_interval = {}

    if not include_projection:
        return observed, global_interval

    projected_rows = []

    for class_id, group in observed.groupby("ClassId", sort=False):
        last = group.sort_values("LactationNumber").iloc[-1]
        last_lactation = int(last["LactationNumber"])

        if last_lactation >= 10:
            continue

        interval = int(
            round(animal_interval.get(int(class_id), global_interval))
        )

        projected_rows.append(
            {
                "AnimalId": last["AnimalId"],
                "LactationNumber": last_lactation + 1,
                "LactationStartDay": int(last["LactationStartDay"] + interval),
                "LastEventDay": int(last["LastEventDay"]),
                "LastDaysInMilk": int(last["LastDaysInMilk"]),
                "ClassId": int(class_id),
                "IsProjected": True,
            }
        )

    if projected_rows:
        profiles = pd.concat(
            [observed, pd.DataFrame(projected_rows)],
            ignore_index=True,
        )
    else:
        profiles = observed.copy()

    return profiles, global_interval


def make_profile_lookup(profiles):
    lookup = {}

    for lactation, group in profiles.groupby("LactationNumber", sort=False):
        lookup[int(lactation)] = {
            "class_ids": group["ClassId"].to_numpy(dtype=np.int32),
            "start_days": group["LactationStartDay"].to_numpy(dtype=np.float32),
            "last_event_days": group["LastEventDay"].to_numpy(dtype=np.float32),
            "last_dim": group["LastDaysInMilk"].to_numpy(dtype=np.float32),
            "projected": group["IsProjected"].to_numpy(dtype=bool),
        }

    return lookup


def get_candidate_profiles(
    profile_lookup,
    lactation_number,
    query_start_day,
    max_candidates=GLOBAL_MAX_CANDIDATES,
    projected_rank_penalty_days=PROJECTED_RANK_PENALTY_DAYS,
):
    """Global structural retrieval: no hard start-date tolerance."""
    if pd.isna(lactation_number) or pd.isna(query_start_day):
        return None

    lactation_number = int(round(float(lactation_number)))
    profile = profile_lookup.get(lactation_number)

    if profile is None:
        return None

    start_difference = np.abs(
        profile["start_days"] - float(query_start_day)
    )

    structural_score = (
        start_difference
        + profile["projected"].astype(np.float32)
        * projected_rank_penalty_days
    )

    positions = np.argsort(structural_score)[
        : min(max_candidates, len(structural_score))
    ]

    return {
        key: value[positions]
        for key, value in profile.items()
    } | {
        "start_difference": start_difference[positions]
    }


def create_distance_features(raw_frame):
    X = raw_frame[
        BASE_NUMERIC_FEATURES + ["ReproductionStatus"]
    ].copy()

    duration_min = X["DurationSession_sec"].astype(np.float32) / 60.0

    X["YieldPerMinute"] = (
        X["YieldSession"] / duration_min.replace(0, np.nan)
    )
    X["First2MinFraction"] = (
        X["YieldFirst2Min_Session"]
        / X["YieldSession"].replace(0, np.nan)
    )
    X["FlowShapeRatio"] = (
        X["Flow30_60Session"]
        / X["Avgmilkflow"].replace(0, np.nan)
    )

    return X


def fit_distance_preprocessor(raw_frame):
    features = create_distance_features(raw_frame)

    numeric_columns = [
        c for c in features.columns
        if c != "ReproductionStatus"
    ]

    numeric = features[numeric_columns].copy()
    numeric_fill = numeric.median(numeric_only=True)
    numeric = numeric.fillna(numeric_fill)

    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(numeric).astype(np.float32)

    reproduction = pd.get_dummies(
        features["ReproductionStatus"],
        prefix="ReproductionStatus",
        dtype=np.float32,
    )

    reproduction_columns = list(reproduction.columns)

    X = np.column_stack(
        [
            numeric_scaled,
            reproduction.to_numpy(dtype=np.float32),
        ]
    ).astype(np.float32)

    feature_names = numeric_columns + reproduction_columns

    return (
        X,
        numeric_columns,
        numeric_fill,
        scaler,
        reproduction_columns,
        feature_names,
    )


def transform_distance_features(
    raw_frame,
    numeric_columns,
    numeric_fill,
    scaler,
    reproduction_columns,
):
    features = create_distance_features(raw_frame)

    numeric = (
        features[numeric_columns]
        .copy()
        .fillna(numeric_fill)
    )

    numeric_scaled = scaler.transform(numeric).astype(np.float32)

    reproduction = (
        pd.get_dummies(
            features["ReproductionStatus"],
            prefix="ReproductionStatus",
            dtype=np.float32,
        )
        .reindex(columns=reproduction_columns, fill_value=0)
    )

    return np.column_stack(
        [
            numeric_scaled,
            reproduction.to_numpy(dtype=np.float32),
        ]
    ).astype(np.float32)


def sample_reference_rows(
    frame,
    max_rows_per_group=REFERENCE_ROWS_PER_GROUP,
    recent_fraction=0.80,
    random_state=RANDOM_STATE,
):
    selected_indices = []

    grouped = frame.groupby(
        ["AnimalId", "LactationNumber"],
        observed=True,
        sort=False,
    )

    for _, group in grouped:
        group = group.sort_values("EventDate")

        if len(group) <= max_rows_per_group:
            selected_indices.extend(group.index.tolist())
            continue

        n_recent = max(
            1,
            int(round(max_rows_per_group * recent_fraction)),
        )

        recent = group.tail(n_recent)
        remaining_n = max_rows_per_group - len(recent)

        if remaining_n > 0:
            historical_pool = group.loc[
                ~group.index.isin(recent.index)
            ]

            historical = historical_pool.sample(
                n=min(remaining_n, len(historical_pool)),
                replace=False,
                random_state=random_state,
            )

            selected = pd.concat([recent, historical])
        else:
            selected = recent

        selected_indices.extend(selected.index.tolist())

    return pd.Index(selected_indices)


def build_reference_indices(class_ids, lactations):
    """Build exact class/lactation, per-class lactation, and class-position indices."""
    exact = {}
    class_lactations = {}
    class_positions = {}

    for row_position, (class_id, lactation) in enumerate(
        zip(class_ids, lactations)
    ):
        class_id = int(class_id)
        lactation = int(lactation)

        exact.setdefault((class_id, lactation), []).append(row_position)
        class_lactations.setdefault(class_id, set()).add(lactation)
        class_positions.setdefault(class_id, []).append(row_position)

    exact = {
        key: np.asarray(positions, dtype=np.int32)
        for key, positions in exact.items()
    }

    class_lactations = {
        class_id: np.asarray(sorted(values), dtype=np.int16)
        for class_id, values in class_lactations.items()
    }

    class_positions = {
        class_id: np.asarray(positions, dtype=np.int32)
        for class_id, positions in class_positions.items()
    }

    return exact, class_lactations, class_positions


def choose_reference_lactation(
    class_id,
    query_lactation,
    class_lactations,
):
    available = class_lactations.get(int(class_id))

    if available is None or len(available) == 0:
        return None, 99

    query_lactation = int(round(float(query_lactation)))

    if query_lactation in available:
        return query_lactation, 0

    previous = available[available <= query_lactation]

    if len(previous) > 0:
        selected = int(previous.max())
    else:
        selected = int(
            available[np.argmin(np.abs(available - query_lactation))]
        )

    return selected, abs(query_lactation - selected)


def get_nearest_lactation_prototype(
    prototype_table,
    class_id,
    lactation_number,
    query_dim,
):
    prototype_dict = prototype_table.get(
        (int(class_id), int(lactation_number))
    )

    if not prototype_dict:
        return None

    query_bin = int(np.floor(float(query_dim) / DIM_BIN_WIDTH))
    available_bins = np.asarray(
        list(prototype_dict.keys()),
        dtype=np.int16,
    )

    nearest_bin = int(
        available_bins[
            np.argmin(np.abs(available_bins - query_bin))
        ]
    )

    return prototype_dict[nearest_bin]


print("Core helper functions loaded.")


In [ ]:

# ============================================================
# 4. Reference-system builder and candidate-feature pipeline
# ============================================================

RANKING_COLUMNS = [
    "StartDifference",
    "RecencyDays",
    "ClosestDIMDifference",
    "LocalMinManhattan",
    "LocalMean3Manhattan",
    "LocalMean5Manhattan",
    "LocalMinEuclidean",
    "LocalMean3Euclidean",
    "LactationPrototypeManhattan",
    "LactationPrototypeEuclidean",
    "CowPrototypeManhattan",
    "CowPrototypeEuclidean",
    "ReproductionDistance",
    "Residual_DaysInMilk",
    "Residual_Avgmilkflow",
    "Residual_Flow30_60Session",
    "Residual_YieldFirst2Min_Session",
    "Residual_YieldSession",
    "Residual_DurationSession_sec",
    "Residual_YieldPerMinute",
    "Residual_First2MinFraction",
    "Residual_FlowShapeRatio",
]

NON_FEATURE_COLUMNS = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]


def build_reference_system(reference_df):
    """Build all snapshot-specific objects in one dictionary.

    Keeping the state inside a dictionary avoids the fragile global-variable
    swapping used in the exploratory notebook.
    """
    class_values = pd.Index(
        reference_df["AnimalId"].astype(object).unique()
    )

    class_to_number = {
        animal_id: class_number
        for class_number, animal_id in enumerate(class_values)
    }

    origin_date = pd.to_datetime(reference_df["EventDate"]).min()

    profiles, global_lactation_interval = build_structural_profiles(
        reference_df,
        class_to_number,
        origin_date,
        include_projection=True,
    )

    profile_lookup = make_profile_lookup(profiles)

    reference_indices = sample_reference_rows(reference_df)

    reference_raw = reference_df.loc[reference_indices].copy()

    (
        X_reference,
        numeric_columns,
        numeric_fill,
        scaler,
        reproduction_columns,
        feature_names,
    ) = fit_distance_preprocessor(reference_raw)

    reference_class_ids = (
        reference_raw["AnimalId"]
        .astype(object)
        .map(class_to_number)
        .astype(np.int32)
        .to_numpy()
    )

    reference_lactations = (
        reference_raw["LactationNumber"]
        .round()
        .astype(np.int16)
        .to_numpy()
    )

    reference_dim = (
        reference_raw["DaysInMilk"]
        .astype(np.float32)
        .to_numpy()
    )

    (
        reference_exact_index,
        reference_class_lactations,
        class_positions,
    ) = build_reference_indices(
        reference_class_ids,
        reference_lactations,
    )

    # Whole-cow prototypes. class_positions avoids repeatedly scanning
    # the entire reference array for each class.
    cow_prototype = {
        class_id: np.median(
            X_reference[positions],
            axis=0,
        ).astype(np.float32)
        for class_id, positions in class_positions.items()
    }

    # Lactation + DIM-bin prototypes.
    reference_dim_bin = np.floor(
        reference_dim / DIM_BIN_WIDTH
    ).astype(np.int16)

    lactation_prototype = {}

    for (class_id, lactation_number), positions in (
        reference_exact_index.items()
    ):
        bins_here = reference_dim_bin[positions]
        prototype_by_bin = {}

        for dim_bin in np.unique(bins_here):
            bin_positions = positions[bins_here == dim_bin]

            prototype_by_bin[int(dim_bin)] = np.median(
                X_reference[bin_positions],
                axis=0,
            ).astype(np.float32)

        lactation_prototype[
            (int(class_id), int(lactation_number))
        ] = prototype_by_bin

    system = {
        "class_values": class_values,
        "class_to_number": class_to_number,
        "origin_date": origin_date,
        "global_lactation_interval": global_lactation_interval,
        "profile_lookup": profile_lookup,
        "X_reference": X_reference,
        "numeric_columns": numeric_columns,
        "numeric_fill": numeric_fill,
        "scaler": scaler,
        "reproduction_columns": reproduction_columns,
        "feature_names": feature_names,
        "reference_dim": reference_dim,
        "reference_exact_index": reference_exact_index,
        "reference_class_lactations": reference_class_lactations,
        "cow_prototype": cow_prototype,
        "lactation_prototype": lactation_prototype,
        "n_reference_rows": len(reference_raw),
        "n_profiles": len(profiles),
    }

    return system


def _nearest_mean(distances, k):
    k = min(k, len(distances))
    return float(
        np.mean(
            np.partition(distances, k - 1)[:k]
        )
    )


def build_candidate_features_for_one_query(
    query_frame,
    query_id,
    ref,
):
    if len(query_frame) != 1:
        raise ValueError("query_frame must contain exactly one row.")

    query_vector = transform_distance_features(
        query_frame,
        ref["numeric_columns"],
        ref["numeric_fill"],
        ref["scaler"],
        ref["reproduction_columns"],
    )[0]

    query_event_day = float(
        event_day_from_origin(
            query_frame,
            ref["origin_date"],
        ).iloc[0]
    )

    query_start_day = float(
        lactation_start_day(
            query_frame,
            ref["origin_date"],
        ).iloc[0]
    )

    query_dim = float(query_frame["DaysInMilk"].iloc[0])
    query_lactation = int(
        round(float(query_frame["LactationNumber"].iloc[0]))
    )

    true_animal_id = query_frame["AnimalId"].astype(object).iloc[0]
    true_class_id = ref["class_to_number"].get(
        true_animal_id,
        np.nan,
    )

    candidate = get_candidate_profiles(
        ref["profile_lookup"],
        query_lactation,
        query_start_day,
    )

    if candidate is None:
        return pd.DataFrame()

    candidate_count = len(candidate["class_ids"])

    numeric_feature_positions = {
        name: position
        for position, name in enumerate(ref["feature_names"])
        if not name.startswith("ReproductionStatus_")
    }

    reproduction_positions = [
        position
        for position, name in enumerate(ref["feature_names"])
        if name.startswith("ReproductionStatus_")
    ]

    rows = []

    for candidate_position in range(candidate_count):
        class_id = int(candidate["class_ids"][candidate_position])
        start_difference = float(
            candidate["start_difference"][candidate_position]
        )
        is_projected = int(
            candidate["projected"][candidate_position]
        )
        last_event_day = float(
            candidate["last_event_days"][candidate_position]
        )

        (
            selected_lactation,
            lactation_gap,
        ) = choose_reference_lactation(
            class_id,
            query_lactation,
            ref["reference_class_lactations"],
        )

        if selected_lactation is None:
            continue

        reference_positions = ref["reference_exact_index"].get(
            (class_id, selected_lactation)
        )

        if reference_positions is None or len(reference_positions) == 0:
            continue

        dim_difference = np.abs(
            ref["reference_dim"][reference_positions] - query_dim
        )

        closest_dim_difference = float(np.min(dim_difference))

        inside_dim = dim_difference <= DIM_TOLERANCE_DAYS

        if inside_dim.any():
            usable_positions = reference_positions[inside_dim]
        else:
            nearest_order = np.argsort(dim_difference)
            usable_positions = reference_positions[
                nearest_order[: min(10, len(nearest_order))]
            ]

        differences = (
            ref["X_reference"][usable_positions] - query_vector
        )

        manhattan_distances = np.sum(
            np.abs(differences),
            axis=1,
        )

        euclidean_distances = np.sqrt(
            np.sum(differences ** 2, axis=1)
        )

        local_min_manhattan = float(np.min(manhattan_distances))
        local_mean3_manhattan = _nearest_mean(manhattan_distances, 3)
        local_mean5_manhattan = _nearest_mean(manhattan_distances, 5)
        local_min_euclidean = float(np.min(euclidean_distances))
        local_mean3_euclidean = _nearest_mean(euclidean_distances, 3)

        lactation_prototype = get_nearest_lactation_prototype(
            ref["lactation_prototype"],
            class_id,
            selected_lactation,
            query_dim,
        )

        residual_features = {}

        if lactation_prototype is not None:
            prototype_difference = query_vector - lactation_prototype

            lactation_manhattan = float(
                np.sum(np.abs(prototype_difference))
            )

            lactation_euclidean = float(
                np.sqrt(np.sum(prototype_difference ** 2))
            )

            for feature_name, position in numeric_feature_positions.items():
                residual_features[
                    f"Residual_{feature_name}"
                ] = float(abs(prototype_difference[position]))

            if reproduction_positions:
                reproduction_distance = float(
                    np.sum(
                        np.abs(
                            prototype_difference[reproduction_positions]
                        )
                    )
                )
            else:
                reproduction_distance = 0.0

        else:
            lactation_manhattan = np.nan
            lactation_euclidean = np.nan
            reproduction_distance = np.nan

            for feature_name in numeric_feature_positions:
                residual_features[
                    f"Residual_{feature_name}"
                ] = np.nan

        cow_prototype = ref["cow_prototype"].get(class_id)

        if cow_prototype is not None:
            cow_difference = query_vector - cow_prototype

            cow_manhattan = float(
                np.sum(np.abs(cow_difference))
            )

            cow_euclidean = float(
                np.sqrt(np.sum(cow_difference ** 2))
            )
        else:
            cow_manhattan = np.nan
            cow_euclidean = np.nan

        recency_days = max(
            0.0,
            query_event_day - last_event_day,
        )

        row = {
            "QueryId": query_id,
            "CandidateClassId": class_id,
            "TrueClassId": true_class_id,
            "IsCorrectCandidate": (
                int(class_id == true_class_id)
                if not pd.isna(true_class_id)
                else 0
            ),
            "CandidateCount": candidate_count,
            "CandidateStartRank": candidate_position + 1,
            "StartDifference": start_difference,
            "IsProjected": is_projected,
            "RecencyDays": recency_days,
            "LogRecencyDays": np.log1p(recency_days),
            "LactationGap": lactation_gap,
            "QueryDaysInMilk": query_dim,
            "QueryLactationNumber": query_lactation,
            "ClosestDIMDifference": closest_dim_difference,
            "ReferenceRowCount": len(reference_positions),
            "LocalMinManhattan": local_min_manhattan,
            "LocalMean3Manhattan": local_mean3_manhattan,
            "LocalMean5Manhattan": local_mean5_manhattan,
            "LocalMinEuclidean": local_min_euclidean,
            "LocalMean3Euclidean": local_mean3_euclidean,
            "LactationPrototypeManhattan": lactation_manhattan,
            "LactationPrototypeEuclidean": lactation_euclidean,
            "CowPrototypeManhattan": cow_manhattan,
            "CowPrototypeEuclidean": cow_euclidean,
            "ReproductionDistance": reproduction_distance,
        }

        row.update(residual_features)
        rows.append(row)

    return pd.DataFrame(rows)


def add_candidate_rank_features(candidate_pairs):
    result = candidate_pairs.copy()

    for column in RANKING_COLUMNS:
        if column not in result.columns:
            continue

        result[f"{column}_Rank"] = (
            result.groupby("QueryId")[column]
            .rank(
                method="average",
                ascending=True,
                pct=True,
                na_option="bottom",
            )
            .astype(np.float32)
        )

    return result


def build_candidate_dataset(
    query_frame,
    prefix,
    ref,
    progress_every=100,
):
    """Build Global Top-320 candidate rows.

    Rank features are added later after Top-160 filtering, which avoids
    computing ranks twice in a clean run.
    """
    tables = []

    for counter, query_index in enumerate(
        query_frame.index,
        start=1,
    ):
        one_query = query_frame.loc[[query_index]].copy()

        candidate_table = build_candidate_features_for_one_query(
            one_query,
            query_id=f"{prefix}_{query_index}",
            ref=ref,
        )

        if len(candidate_table) > 0:
            tables.append(candidate_table)

        if (
            counter % progress_every == 0
            or counter == len(query_frame)
        ):
            print(
                f"{prefix}: processed "
                f"{counter} / {len(query_frame)} queries"
            )

    if not tables:
        return pd.DataFrame()

    return pd.concat(tables, ignore_index=True)


def prepare_topk_for_model(
    candidate_pairs,
    top_k=MODEL_TOP_K,
    require_positive=False,
):
    result = (
        candidate_pairs.loc[
            candidate_pairs["CandidateStartRank"] <= top_k
        ]
        .copy()
        .reset_index(drop=True)
    )

    result["CandidateCount"] = (
        result.groupby("QueryId")["QueryId"]
        .transform("size")
        .astype(np.int16)
    )

    # Recompute ranks inside the actual Top-K model candidate set.
    result = add_candidate_rank_features(result)

    if require_positive:
        positive_available = (
            result.groupby("QueryId")["IsCorrectCandidate"]
            .transform("max")
        )
        result = (
            result.loc[positive_available == 1]
            .copy()
            .reset_index(drop=True)
        )

    return result


def load_or_build_snapshot_candidates(
    reference_df,
    query_df,
    prefix,
    cache_path,
    progress_every=100,
):
    expected_ids = {
        f"{prefix}_{idx}"
        for idx in query_df.index
    }

    if cache_path.exists():
        cached = pd.read_pickle(cache_path)
        cached_ids = set(cached["QueryId"].unique())

        # Allow a tiny fraction of zero-candidate queries while rejecting
        # an obviously stale cache from a different query plan.
        if (
            cached_ids.issubset(expected_ids)
            and len(cached_ids) >= 0.95 * len(expected_ids)
        ):
            print("Loaded cache:", cache_path.name)
            return cached

        print("Cache/query mismatch; rebuilding:", cache_path.name)

    ref = build_reference_system(reference_df)

    print(
        f"Reference: {len(reference_df):,} rows | "
        f"{len(ref['class_values']):,} AnimalIds | "
        f"{ref['n_reference_rows']:,} sampled rows | "
        f"matrix {ref['X_reference'].shape}"
    )

    candidate_pairs = build_candidate_dataset(
        query_df,
        prefix=prefix,
        ref=ref,
        progress_every=progress_every,
    )

    candidate_pairs.to_pickle(cache_path)
    print("Saved cache:", cache_path.name)

    del ref
    gc.collect()

    return candidate_pairs


print("Reference and candidate pipeline loaded.")


In [ ]:
def load_labelled_data(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f'Cannot find labelled data: {path}')
    frame = pd.read_csv(path, usecols=lambda c: c.strip() in MODEL_COLUMNS, low_memory=False)
    frame.columns = frame.columns.astype(str).str.strip()
    missing = set(MODEL_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f'Labelled CSV missing columns: {sorted(missing)}')
    frame = frame[MODEL_COLUMNS].copy()
    frame['AnimalId'] = frame['AnimalId'].astype('string')
    frame['EventDate'] = pd.to_datetime(frame['EventDate'], errors='coerce')
    for col in ['LactationNumber','DaysInMilk','DurationSession_sec']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
    for col in MILK_OBSERVED_COLS[:-1]:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
    frame['ReproductionStatus'] = frame['ReproductionStatus'].astype('category')
    frame = (frame.dropna(subset=['AnimalId', 'EventDate', 'LactationNumber', 'DaysInMilk'])
             .sort_values('EventDate').reset_index(drop=True))
    if frame.empty:
        raise ValueError('No valid labelled reference rows in training CSV')
    return frame


def observable_milk_features(frame, origin):
    # Same 3 milk ratios as the original create_distance_features.
    X = frame[MILK_OBSERVED_COLS].copy()
    for c in MILK_OBSERVED_COLS:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    minutes = X['DurationSession_sec'].astype(np.float32) / 60.0
    X['YieldPerMinute'] = X['YieldSession'] / minutes.replace(0, np.nan)
    X['First2MinFraction'] = X['YieldFirst2Min_Session'] / X['YieldSession'].replace(0, np.nan)
    X['FlowShapeRatio'] = X['Flow30_60Session'] / X['Avgmilkflow'].replace(0, np.nan)
    dates = pd.to_datetime(frame['EventDate'], errors='coerce')
    X['EventDay'] = ((dates - origin).dt.days).astype(np.float32)
    season = (dates.dt.dayofyear - 1) / 365.25
    X['SeasonSin'] = np.sin(2 * np.pi * season)
    X['SeasonCos'] = np.cos(2 * np.pi * season)
    return X.replace([np.inf, -np.inf], np.nan).astype(np.float32)


def fit_auxiliary_models(reference: pd.DataFrame, max_rows: int):
    if len(reference) > max_rows:
        train = reference.sample(n=max_rows, random_state=RANDOM_STATE)
    else:
        train = reference
    origin = reference['EventDate'].min()
    X = observable_milk_features(train, origin)
    known_lac = pd.to_numeric(train['LactationNumber'], errors='coerce').round()
    valid_lac = known_lac.between(1, 10)
    if valid_lac.sum() < 100 or known_lac.loc[valid_lac].nunique() < 2:
        raise ValueError('Not enough valid lactation labels to fit imputer')
    lac_model = HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=140, max_leaf_nodes=25,
        min_samples_leaf=40, random_state=RANDOM_STATE)
    lac_model.fit(X.loc[valid_lac], known_lac.loc[valid_lac].astype(np.int16))

    known_dim = pd.to_numeric(train['DaysInMilk'], errors='coerce')
    valid_dim = known_dim.between(1, 2000)
    if valid_dim.sum() < 100:
        raise ValueError('Not enough valid DaysInMilk labels to fit imputer')
    dim_upper = float(np.nanpercentile(known_dim.loc[valid_dim], 99.9))
    dim_model = HistGradientBoostingRegressor(
        learning_rate=0.07, max_iter=180, max_leaf_nodes=31,
        min_samples_leaf=40, random_state=RANDOM_STATE)
    dim_model.fit(X.loc[valid_dim], known_dim.loc[valid_dim].astype(np.float32))

    known_status = train['ReproductionStatus'].astype('string')
    status_counts = known_status.dropna().value_counts()
    status_fallback = str(status_counts.index[0]) if len(status_counts) else 'Unknown'
    common_status = status_counts.index[status_counts >= 30]
    valid_status = known_status.isin(common_status).fillna(False)
    status_model = None
    if known_status.loc[valid_status].nunique() >= 2:
        status_model = HistGradientBoostingClassifier(
            learning_rate=0.08, max_iter=100, max_leaf_nodes=15,
            min_samples_leaf=50, random_state=RANDOM_STATE)
        status_model.fit(X.loc[valid_status], known_status.loc[valid_status].astype(str))
    print(f'Auxiliary models fitted on {len(train):,} labelled rows', flush=True)
    return lac_model, dim_model, status_model, status_fallback, dim_upper, origin


def load_feature_order(model, feature_json: Path|None, ref):
    # Original final HGB was fitted on a pandas DataFrame, which preserves the
    # exact feature names AND order in sklearn's feature_names_in_.
    feature_names = getattr(model, 'feature_names_in_', None)
    if feature_names is not None:
        names = list(feature_names)
        print('Feature order loaded from frozen model.feature_names_in_', flush=True)
    elif feature_json is not None and feature_json.exists():
        with feature_json.open(encoding='utf-8') as f:
            names = json.load(f)
        print('Feature order loaded from optional JSON', flush=True)
    else:
        # Derive the same deterministic feature order as original prepare_topk_for_model.
        # Used only when an older sklearn model lacks feature_names_in_.
        names = None
        for offset in range(0, min(len(ref['probe_data']), 500), 20):
            frame = ref['probe_data'].iloc[offset:offset+20]
            with redirect_stdout(io.StringIO()):
                candidates = build_candidate_dataset(frame, prefix='probe', ref=ref, progress_every=10000)
            if candidates.empty:
                continue
            top160 = prepare_topk_for_model(candidates, top_k=MODEL_TOP_K, require_positive=False)
            if not top160.empty:
                names = [c for c in top160.columns if c not in NON_FEATURE_COLUMNS]
                break
        if names is None:
            raise ValueError('Could not infer model feature names; supply --features JSON')
        print('Feature order derived from original candidate-feature pipeline', flush=True)
    if len(names) != 52 or getattr(model, 'n_features_in_', 52) != len(names):
        raise ValueError(f'Frozen HGB feature count does not match 52: {len(names)}')
    return names


def make_predictor(reference, frozen_model, features, max_aux_rows):
    lac_imputer, dim_imputer, status_imputer, status_fallback, dim_upper, origin = (
        fit_auxiliary_models(reference, max_aux_rows)
    )
    print('Building original reference system...', flush=True)
    ref = build_reference_system(reference)
    ref_lactations = np.array(sorted(ref['profile_lookup'].keys()), dtype=np.int16)
    if not len(ref_lactations):
        raise ValueError('No reference lactation profiles')
    print(f'Reference: {len(reference):,} rows, {len(ref["class_values"]):,} animals, {ref["n_reference_rows"]:,} sampled rows', flush=True)
    milk_distance_features = MILK_OBSERVED_COLS + ['YieldPerMinute', 'First2MinFraction', 'FlowShapeRatio']
    milk_positions = [ref['feature_names'].index(col) for col in milk_distance_features]
    ref_row_classes = np.full(len(ref['X_reference']), -1, dtype=np.int32)
    for (class_num, _), positions in ref['reference_exact_index'].items():
        ref_row_classes[positions] = int(class_num)
    if (ref_row_classes < 0).any():
        raise ValueError('Reference row-to-animal lookup was incomplete')
    milk_neighbor = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(
        ref['X_reference'][:, milk_positions]
    )

    def predict_chunk(raw_chunk: pd.DataFrame, start: int) -> pd.DataFrame:
        query = raw_chunk.copy()
        query.index = pd.RangeIndex(start, start + len(query))
        query['AnimalId'] = pd.Series(pd.NA, index=query.index, dtype='string')
        query['EventDate'] = pd.to_datetime(query['EventDate'], errors='coerce')
        for col in MILK_OBSERVED_COLS:
            query[col] = pd.to_numeric(query[col], errors='coerce')
        visible_X = observable_milk_features(query, origin)
        lac_est = lac_imputer.predict(visible_X).astype(np.int16)
        closest = np.abs(lac_est[:, None] - ref_lactations[None, :]).argmin(axis=1)
        query['LactationNumber'] = ref_lactations[closest]
        query['DaysInMilk'] = np.rint(np.clip(dim_imputer.predict(visible_X), 1, max(1, dim_upper))).astype(np.int16)
        statuses = (status_imputer.predict(visible_X) if status_imputer is not None
                    else np.repeat(status_fallback, len(query)))
        query['ReproductionStatus'] = pd.Categorical(statuses)
        predictions = pd.Series(pd.NA, index=query.index, dtype='string')

        with redirect_stdout(io.StringIO()):
            top320 = build_candidate_dataset(query, prefix='practice', ref=ref, progress_every=max(1,len(query)))
        if not top320.empty:
            top160 = prepare_topk_for_model(top320, top_k=MODEL_TOP_K, require_positive=False)
            if not top160.empty:
                absent = [c for c in features if c not in top160.columns]
                if absent:
                    raise ValueError(f'Missing HGB input features: {absent}')
                x = top160[features].replace([np.inf, -np.inf], np.nan).astype(np.float32)
                scores = frozen_model.predict_proba(x)[:,1]
                scored = top160[['QueryId', 'CandidateClassId']].copy()
                scored['Probability'] = scores
                winners = scored.loc[scored.groupby('QueryId',sort=False)['Probability'].idxmax()]
                mapped = pd.Series(ref['class_values'].take(winners['CandidateClassId'].astype(np.int32).to_numpy()).astype(str),
                                   index=winners['QueryId'])
                matched = pd.Index([f'practice_{i}' for i in query.index]).map(mapped)
                predictions.loc[:] = pd.array(matched, dtype='string')

        # Always provide a prediction: original structural candidates may be empty
        # when the estimated lactation start is outside the reference range.
        missing = predictions.isna()
        if missing.any():
            missing_rows = query.loc[missing]
            x_nearest = transform_distance_features(
                missing_rows, ref['numeric_columns'], ref['numeric_fill'], ref['scaler'],
                ref['reproduction_columns'])[:, milk_positions]
            _, neighbor_rows = milk_neighbor.kneighbors(x_nearest)
            classes = ref_row_classes[neighbor_rows.ravel()]
            predictions.loc[missing] = pd.array(ref['class_values'].take(classes).astype(str), dtype='string')

        output = pd.DataFrame({'AnimalId': predictions,
                               'MilkingId': query['MilkingId'].astype('string')}, index=query.index)
        output = output[['AnimalId', 'MilkingId']]
        if len(output) != len(raw_chunk) or output['AnimalId'].isna().any():
            raise RuntimeError('Prediction output lost rows or contains blank AnimalId')
        return output
    return predict_chunk


def create_csv(train_path, test_path, model_path, output_path, feature_path,
               chunk_size=300, max_aux_rows=250000, reset_cache=False):
    for p in (test_path, model_path):
        if not p.is_file():
            raise FileNotFoundError(f'Required file not found: {p}')
    if chunk_size < 1:
        raise ValueError('Chunk size must be >= 1')
    with test_path.open(encoding='utf-8-sig', newline='') as f:
        columns = next(csv.reader(f))
    missing = (set(MODEL_COLUMNS) | {'MilkingId'}) - set(columns)
    if missing:
        raise ValueError(f'Practice test missing columns: {sorted(missing)}')
    print('Loading labelled reference and frozen model...', flush=True)
    reference = load_labelled_data(train_path)
    frozen = joblib.load(model_path)
    if not hasattr(frozen, 'predict_proba'):
        raise ValueError('This file is not a fitted classifier with predict_proba')
    predictor_ref = {'probe_data': reference}
    # If needed, compute reference before deriving feature names only if model
    # lacks feature_names_in_. A typical final trained HGB needs no JSON.
    if getattr(frozen, 'feature_names_in_', None) is None and not (feature_path and feature_path.exists()):
        temp_ref = build_reference_system(reference)
        predictor_ref.update(temp_ref)
    features = load_feature_order(frozen, feature_path, predictor_ref)
    predictor = make_predictor(reference, frozen, features, max_aux_rows)

    parts_folder = output_path.parent / (output_path.stem + '_checkpoint_parts')
    if reset_cache and parts_folder.exists():
        import shutil
        shutil.rmtree(parts_folder)
    parts_folder.mkdir(parents=True, exist_ok=True)
    stat_train, stat_test, stat_model = train_path.stat(), test_path.stat(), model_path.stat()
    signature = {
        'version': 'original_top320_top160_HGB_aux_v2',
        'train': str(train_path.resolve()), 'train_size':stat_train.st_size,'train_mtime_ns':stat_train.st_mtime_ns,
        'test': str(test_path.resolve()), 'test_size':stat_test.st_size,'test_mtime_ns':stat_test.st_mtime_ns,
        'model':str(model_path.resolve()),'model_size':stat_model.st_size,'model_mtime_ns':stat_model.st_mtime_ns,
        'features_sha256':hashlib.sha256(json.dumps(features).encode('utf-8')).hexdigest(),
        'chunk_size':chunk_size, 'max_aux_rows':max_aux_rows,
    }
    manifest = parts_folder / 'manifest.json'
    if manifest.exists():
        with manifest.open(encoding='utf-8') as f:
            if json.load(f) != signature:
                raise RuntimeError(f'Old checkpoint settings do not match. Use --reset-cache or delete {parts_folder}')
    else:
        if list(parts_folder.glob('chunk_*.csv')):
            raise RuntimeError(f'Untracked cached prediction parts: {parts_folder}. Use --reset-cache')
        with manifest.open('w', encoding='utf-8') as f:
            json.dump(signature, f, indent=2)

    columns_to_read = MODEL_COLUMNS + ['MilkingId']
    total, parts = 0, []
    reader = pd.read_csv(test_path, usecols=columns_to_read, dtype={'AnimalId':'string','MilkingId':'string'},
                         chunksize=chunk_size, low_memory=False)
    for part_no, raw_chunk in enumerate(reader):
        file = parts_folder / f'chunk_{part_no:06d}.csv'
        parts.append((file,len(raw_chunk)))
        if not file.exists():
            output = predictor(raw_chunk, total)
            tmp = file.with_suffix('.tmp')
            output.to_csv(tmp, index=False)
            os.replace(tmp, file)
        total += len(raw_chunk)
        if (part_no + 1) % 25 == 0:
            print(f'Processed/reused {total:,} test rows', flush=True)

    temporary_output = output_path.with_suffix('.partial.csv')
    written = 0
    with temporary_output.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['AnimalId','MilkingId'])
        for file, nrows in parts:
            n = 0
            with file.open(encoding='utf-8-sig', newline='') as part_f:
                rows = csv.DictReader(part_f)
                if rows.fieldnames != ['AnimalId','MilkingId']:
                    raise ValueError(f'Unexpected checkpoint columns: {file}')
                for row in rows:
                    if not row['AnimalId'].strip():
                        raise ValueError(f'Blank AnimalId in {file}')
                    writer.writerow([row['AnimalId'],row['MilkingId']])
                    n += 1
            if n != nrows:
                raise ValueError(f'Checkpoint row count mismatch: {file}: {n} vs {nrows}')
            written += n
    if written != total:
        raise AssertionError(f'Row mismatch {written} vs {total}')
    os.replace(temporary_output, output_path)
    print(f'DONE: {output_path.resolve()}', flush=True)
    print(f'Rows: {written:,}; columns: AnimalId,MilkingId', flush=True)




In [ ]:
# Adjust ROOT only if your CSV/model files are located elsewhere.
ROOT = Path.cwd()
TRAIN_CSV = ROOT / 'Dataset_for_model_buliding_2.csv'
TEST_CSV = ROOT / 'PracticeTest.csv'
FROZEN_MODEL = ROOT / 'xgboost_cache' / 'hybrid_hgb_top160_final.joblib'
FEATURE_JSON = ROOT / 'xgboost_cache' / 'hybrid_hgb_top160_features.json'  # optional
OUTPUT_CSV = ROOT / 'PracticeTest_AnimalId_predictions.csv'

create_csv(
    train_path=TRAIN_CSV,
    test_path=TEST_CSV,
    model_path=FROZEN_MODEL,
    output_path=OUTPUT_CSV,
    feature_path=FEATURE_JSON,
    chunk_size=300,
    max_aux_rows=250000,
)
